# 04 — Baseline Comparison (RQ3 / Figure 5)

Reproduces the QALIS vs baseline coverage/effectiveness comparison (Figure 5)
and the Wilcoxon signed-rank statistical tests.

**Baselines**: ISO/IEC 25010 (adapted), HELM, MLflow  
**Result**: QALIS significantly outperforms all three baselines on all six dimensions  
  (Wilcoxon signed-rank, all p < 0.001, Bonferroni-corrected α = 0.000556)


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

df = pd.read_csv('../baselines/comparative_analysis_full.csv')
print(df.shape)
print(df.head(8).to_string(index=False))

(96, 8)
 system_id dimension   approach  coverage_score  effectiveness_score  defects_detected  wilcoxon_p_value  bonferroni_sig
        S1        FC      QALIS            0.91                0.89                 9            0.000041           True
        S1        FC   ISO25010            0.69                0.67                 6            0.000041           True
        S1        FC       HELM            0.76                0.74                 7            0.000041           True
        S1        FC     MLflow            0.53                0.51                 5            0.000041           True
        S1        RO      QALIS            0.83                0.81                 8            0.000027           True
        S1        RO   ISO25010            0.56                0.54                 5            0.000027           True


## 1. Figure 5 — Coverage & effectiveness bar chart

In [ ]:
dims        = ['FC','RO','SF','SS','TI','IQ']
approaches  = ['QALIS','ISO25010','HELM','MLflow']
colors      = {'QALIS':'#2ecc71','ISO25010':'#e74c3c','HELM':'#3498db','MLflow':'#f39c12'}

# Mean effectiveness score across all 4 systems per dimension × approach
eff = df.groupby(['dimension','approach'])['effectiveness_score'].mean().unstack()

x     = np.arange(len(dims))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 6))
for i, ap in enumerate(approaches):
    vals = [eff.loc[d, ap] if d in eff.index and ap in eff.columns else 0 for d in dims]
    bars = ax.bar(x + i*width, vals, width, label=ap, color=colors[ap],
                  alpha=0.88, edgecolor='white', linewidth=0.5)

# Significance stars over QALIS bars
for j, d in enumerate(dims):
    qalis_val = eff.loc[d,'QALIS'] if d in eff.index else 0
    ax.text(x[j] + 0*width, qalis_val + 0.015, '★',
            ha='center', va='bottom', fontsize=11, color='darkgreen', fontweight='bold')

ax.axhline(0.5, color='grey', linestyle=':', linewidth=0.8, label='Chance (0.5)')
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(dims, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Effectiveness Score', fontsize=12)
ax.set_title('Figure 5 — QALIS vs Baseline Effectiveness by Dimension\n'
             '★ = QALIS significantly superior (Wilcoxon, all p < 0.001, Bonferroni-corrected)',
             fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/figures/figure5_comparative_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → docs/figures/figure5_comparative_effectiveness.png")

Saved → docs/figures/figure5_comparative_effectiveness.png


## 2. Summary table — mean effectiveness scores

In [ ]:
summary = eff[approaches].loc[dims].round(3)
summary.index.name = 'Dimension'
print(summary.to_string())
print()
# Compute QALIS advantage over each baseline
for ap in approaches[1:]:
    delta = (eff['QALIS'] - eff[ap]).loc[dims]
    print(f"QALIS advantage vs {ap:10}: mean Δ={delta.mean():.3f}  max Δ={delta.max():.3f} ({delta.idxmax()})")

Dimension    QALIS  ISO25010   HELM  MLflow
FC           0.890     0.670  0.740   0.510
RO           0.810     0.540  0.690   0.430
SF           0.870     0.310  0.620   0.280
SS           0.910     0.580  0.490   0.610
TI           0.780     0.290  0.380   0.190
IQ           0.840     0.710  0.220   0.770

QALIS advantage vs ISO25010  : mean Δ=0.368  max Δ=0.490 (TI)
QALIS advantage vs HELM      : mean Δ=0.290  max Δ=0.620 (IQ)
QALIS advantage vs MLflow    : mean Δ=0.388  max Δ=0.590 (TI)


## 3. Wilcoxon test results

In [ ]:
wilc = json.load(open('../experiments/rq3_comparative_effectiveness/exp_rq3_001_wilcoxon_tests.json'))

print(f"Statistical design:")
print(f"  Test:   {wilc['statistical_design']['test']}")
print(f"  α:      {wilc['statistical_design']['alpha']}")
print(f"  k comparisons: {wilc['statistical_design']['n_comparisons']}")
print(f"  Bonferroni α:  {wilc['statistical_design']['bonferroni_corrected_alpha']}")
print()
print(f"{'Dimension':<12} {'Baseline':<12} {'W statistic':>12} {'p-value':>14} {'Sig?':>6}")
print('-' * 58)
for c in wilc['results']['comparisons']:
    print(f"{c['dimension']:<12} {c['baseline']:<12} {c['W']:>12,} {c['p']:>14.6f} {'✓' if c['sig'] else '✗':>6}")
print()
print(f"All significant: {wilc['results']['all_significant']}")
print(f"Largest effect:  {wilc['results']['largest_effect_pair']}")

Statistical design:
  Test:   Wilcoxon signed-rank (one-sided, QALIS > baseline)
  α:      0.01
  k comparisons: 18
  Bonferroni α:  0.000556

Dimension    Baseline        W statistic        p-value   Sig?
----------------------------------------------------------
FC           ISO_25010         4,821,203       0.000041      ✓
FC           HELM              3,912,847       0.000118      ✓
FC           MLflow            5,102,938       0.000009      ✓
RO           ISO_25010         4,987,312       0.000027      ✓
RO           HELM              4,123,091       0.000093      ✓
RO           MLflow            5,301,847       0.000004      ✓
SF           ISO_25010         6,218,473       0.000000      ✓
SF           HELM              4,918,203       0.000031      ✓
SF           MLflow            6,419,823       0.000000      ✓
SS           ISO_25010         4,712,938       0.000058      ✓
SS           HELM              5,819,473       0.000003      ✓
SS           MLflow            4,501,847  

## 4. Baseline blind-spot summary

In [ ]:
blind_spots = {
    'ISO 25010': {'SF': 0.31, 'TI': 0.29,
                  'note': 'No hallucination or transparency metrics'},
    'HELM':      {'IQ': 0.22, 'TI': 0.38,
                  'note': 'Model-centric — no integration quality; TI underserved'},
    'MLflow':    {'SF': 0.28, 'TI': 0.19,
                  'note': 'Infrastructure focus — near-zero semantic and transparency coverage'},
}
print("Baseline blind spots (effectiveness score < 0.40):\n")
for ap, info in blind_spots.items():
    note = info.pop('note')
    gaps = ', '.join(f"{d}={v}" for d,v in info.items())
    print(f"  {ap}: {gaps}")
    print(f"    → {note}")
    print()

Baseline blind spots (effectiveness score < 0.40):

  ISO 25010: SF=0.31, TI=0.29
    → No hallucination or transparency metrics

  HELM: IQ=0.22, TI=0.38
    → Model-centric — no integration quality; TI underserved

  MLflow: SF=0.28, TI=0.19
    → Infrastructure focus — near-zero semantic and transparency coverage

